# Expectation propagation — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def normalize(v):
    v=np.asarray(v,dtype=float); return v/v.sum()
def softmax(v):
    v=np.asarray(v,dtype=float); e=np.exp(v-v.max()); return e/e.sum()
def norm_pdf(x,mu,var):
    return np.exp(-0.5*(np.asarray(x)-mu)**2/var)/np.sqrt(2*np.pi*var)
def beta_pdf_grid(x,a,b):
    B=math.gamma(a)*math.gamma(b)/math.gamma(a+b)
    return (np.asarray(x)**(a-1))*((1-np.asarray(x))**(b-1))/B
print('setup ok')

## Approximate each difficult factor by a simple site and match moments locally
Expectation propagation refines local approximating factors by removing a site, matching tilted moments, and replacing the site. These examples use Gaussian natural parameters to show cavity formation, moment matching, damping, and global approximation.

In [ ]:
# 1) combine Gaussian sites by adding natural parameters
sites=np.array([[1/2,0.0],[1/3,1/3]])  # precision, precision*mean
nat=sites.sum(0); var=1/nat[0]; mean=var*nat[1]
plt.figure(figsize=(6,3)); plt.bar(['mean','variance'],[mean,var]); plt.title('global Gaussian from site products')
assert abs(mean-0.4)<1e-12 and abs(var-1.2)<1e-12

In [ ]:
# 2) cavity distribution removes one site
cav=nat-sites[1]; cav_var=1/cav[0]; cav_mean=cav_var*cav[1]
plt.figure(figsize=(6,3)); plt.bar(['cavity mean','cavity var'],[cav_mean,cav_var]); plt.title('divide out the old site')
assert abs(cav_mean-0.0)<1e-12 and abs(cav_var-2.0)<1e-12

In [ ]:
# 3) moment-matched tilted distribution supplies replacement moments
tilt_mean,tilt_var=0.8,0.5; tilt_nat=np.array([1/tilt_var,tilt_mean/tilt_var]); new_site=tilt_nat-cav
plt.figure(figsize=(6,3)); plt.bar(['site precision','site precision×mean'],new_site); plt.title('new site = tilted minus cavity')
assert np.allclose(new_site,[1.5,1.6])

In [ ]:
# 4) damping prevents violent site jumps
old=sites[1]; damp=0.5*old+0.5*new_site
plt.figure(figsize=(6,3)); plt.plot(old,'o-',label='old'); plt.plot(new_site,'x--',label='new'); plt.plot(damp,'s-',label='damped'); plt.legend(); plt.title('damped EP update')
assert np.all(damp>old)

In [ ]:
# 5) updated global approximation is site product again
nat2=sites[0]+damp; mean2=nat2[1]/nat2[0]; var2=1/nat2[0]
plt.figure(figsize=(6,3)); plt.bar(['old mean','new mean'],[mean,mean2]); plt.title('local update changes global belief')
assert mean2>mean